In [ ]:
from crewai.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field

class SearchInput(BaseModel):
    query: str = Field(..., description="The search query for the tool")
    limit: int = Field(5, description="Number of results")


class CustomSearchTool(BaseTool):
    name: str = "Search Tool"
    description: str = "Searches the web for given query"
    args_schema: Type[BaseModel] = SearchInput  # Link Pydantic here

    def _run(self, query: str, limit: int) -> str:
        # Implementation using query and limit
        return f"Results for {query}: ..."


from crewai import Agent
researcher = Agent(
    role='Researcher',
    goal='Find info',
    backtrace='...',
    tools=[CustomSearchTool()]
)


#### Memory Implementation

In [1]:
import json
import builtins
from crewai import Agent, Task, Crew, Process, LLM, Memory
from crewai.tools import BaseTool, tool
from pydantic import BaseModel,TypeAdapter, Field
from typing import List, Dict
from com.example.agentic.tools.config import create_memory

#
memory = create_memory()

# 1. Define the individual object structure
class Item(BaseModel):
    id: int = Field(..., description="The id of the item")
    name: str = Field(..., description="The name of the item")
    price: float = Field(gt=0, description="The price of the item")
    quantity: int = Field(gt=0, description="the quantity of the item and quantity must be greater than zero")

# 2. Define the schema for the tool's arguments
class OrdersInput(BaseModel):
    records: List[Item] = Field(..., description="A list of items to process")

# Define a Pydantic model for the output structure
class AnalysisOutput(BaseModel):
    total_orders: int
    top_product: str
    total_quantity: int

@tool("Order Analysis Tool")
def order_analysis_tool(records: list) -> dict:
    """Tool to perform order analysis."""
    #_json = json.loads(orders)
    _total_orders=0
    _total_quantity = 0
    _top_product = 'Laptop'
    _ordersummery = {}
    if isinstance(records, list):
        for _order in records :
            _total_orders = _total_orders + 1
            _total_quantity = _total_quantity + _order['quantity']
    else:
        print("It's a JSON object (dictionary) or other type.")
    #
    _ordersummery['total_orders']=_total_orders
    _ordersummery['total_quantity']=_total_quantity
    _ordersummery['top_product']=_top_product
    return _ordersummery

# 3. Create the tool
class OrderAnalysisTool(BaseTool):
    name: str = "Order Analysis Tool"
    description: str = "Tool to perform order analysis."
    args_schema: type[BaseModel] = OrdersInput   # Link Pydantic here

    def _run(self, records: List[Item]) -> str:
        # 4. Access the list of records directly
        records_count = len(records) 
        t_quantity = 0
        t_sell   = 0
        for r in records :
            t_quantity = t_quantity + r.quantity
            t_sell   = t_sell + (r.quantity * r.price)
        
        _total_items = builtins.sum(r.quantity for r in records)
        _top_product = builtins.max(records, key=lambda r: r.quantity)
        #else:
        #    print("It's a JSON object (dictionary) or other type.")
        
        #
        _output = f"Order Summary Report \n\ntotal_orders: {records_count} \ntotal_sell: {t_sell} \ntop_product: {_top_product.name} \ntotal_quantity: {t_quantity}"
        memory.remember(_output)
        return _output

# Test OrderAnalysisTool Tool
# OrderAnalysisTool().run([
#     Item(id=1, name="Laptop", price=1500.00, quantity=4),
#     Item(id=2, name="Moniter", price=1600.00, quantity=8),
#     Item(id=3, name="CPU", price=250.00, quantity=16),
#     Item(id=4, name="RAM", price=150.00, quantity=12)
# ])   

# Create an agent to collect orders
order_collector = Agent(
    role='Order Collector',
    goal='Collect the list of orders from provided orders {orders}.',
    backstory="You are an expert in processing input order data.",
    max_iter=1,
    verbose=True,
    tools=[],
    memory=memory #.scope("/agent/order_collector"),
    #llm=LLM(model="ollama/llama3.2:3b-instruct-q8_0", base_url="http://localhost:11434")
)

# Create an agent to analyze orders
order_analyst = Agent(
    role='Order Analyst',
    goal='Analyze the list of orders and provide a oder summary report.',
    backstory="You are an expert in processing and analyzing order data.",
    tools=[OrderAnalysisTool()],
    max_iter=1,
    verbose=True,
    memory=memory #.scope("/agent/order_analyst"),
    #llm=LLM(model="ollama/llama3.2:3b-instruct-q8_0", base_url="http://localhost:11434")
)

# Define the task using the Pydantic models for input and output validation
order_collector_task = Task(
    description="Analyze the incoming list of orders {orders}.",
    expected_output="provide a pydentic object with  orders attribute that represent list of orders, " \
                    "make sure pydentic object include following attributes :"
                    "- orders",
    output_pydantic=OrdersInput,  # Pydantic model for output structure
    agent=order_collector
)

order_analysis_task = Task(
    description="Analyze the incoming orders from order_collector_task and provide a summary report.",
    expected_output="summary report should include 'total_orders', 'top_product', and 'total_quantity' fields in final report.",
    #input_model=OrdersInput,  # Pydantic model for input validation
    output_pydantic=AnalysisOutput,  # Pydantic model for output structure
    #tools=[OrderAnalysisTool()],
    context=[order_collector_task],
    agent=order_analyst
)

# Define the crew and process
order_analysis_crew = Crew(
    agents=[
        order_collector,
        order_analyst
    ],
    tasks=[
        order_collector_task,
        order_analysis_task
    ],
    process=Process.sequential,
    verbose=True,
    memory=memory,
    #llm=LLM(model="ollama/llama3.2:3b-instruct-q8_0", base_url="http://localhost:11434")
)

#
from datetime import datetime
run_date = datetime.now().strftime('%Y-%m-%d')
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Sample input data (list of orders)
input_data = {
    "orders":[
        {"id": 1, "name":"Laptop", "price":150.00, "quantity": 4},
        {"id": 2, "name":"Monitor", "price":100.5, "quantity": 5},
        {"id": 3, "name":"Laptop", "price":150.00, "quantity": 3},
        {"id": 4, "name":"Monitor", "price":100.5, "quantity": 4},
    ],
    "run_date":run_date,
    "run_id":run_id
}

# Kick off the task
print(f"order analysis crew triggered on {run_date} with execution id {run_id}")
result = order_analysis_crew.kickoff(inputs=input_data)

# The result will be validated and structured by the output_model
print(result)

# Access the Pydantic object
# structured_data = processing_task.output.pydantic # returns a RecordList instance

# Iterate through your records
# for record in structured_data.items:
#     print(f"Processing: {record.name}")

order analysis crew triggered on 2026-04-29 with execution id 20260429_082302


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a5d06fc5-9b34-41a1-a15c-aa96d26fb615                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Invalid type Memory for attribute 'crew_memory' value. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or a sequence of those types


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the incoming list of orders [{'id': 1, 'name': 'Laptop', 'price': 150.0, 'quantity': 4}, {'id':  │
│  2, 'name': 'Monitor', 'price': 100.5, 'quantity': 5}, {'id': 3, 'name': 'Laptop', 'price': 150.0, 'quantity':  │
│  3}, {'id': 4, 'name': 'Monitor', 'price': 100.5, 'quantity': 4}].                                              │
│  ID: 3663fc2a-d010-4dd1-8141-8b3f250c66b5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a5d06fc5-9b34-41a1-a15c-aa96d26fb615                                                                       │
│  Final Output: {"total_orders":150,"top_product":"Laptop","total_quantity":27}                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 34928.09ms                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

total_orders=150 top_product='Laptop' total_quantity=27


In [ ]:
from crewai import Memory
# https://docs.crewai.com/en/concepts/memory
memory = Memory()

# Build up knowledge
memory.remember("The API rate limit is 1000 requests per minute.")
memory.remember("Our staging environment uses port 8080.")
memory.remember("The team agreed to use feature flags for all new releases.")

# Later, recall what you need
matches = memory.recall("What are our API limits?", limit=5)
for m in matches:
    print(f"[{m.score:.2f}] {m.record.content}")

# Extract atomic facts from a longer text
raw = """Meeting notes: We decided to migrate from MySQL to PostgreSQL
next quarter. The budget is $50k. Sarah will lead the migration."""

facts = memory.extract_memories(raw)
# ["Migration from MySQL to PostgreSQL planned for next quarter",
#  "Database migration budget is $50k",
#  "Sarah will lead the database migration"]

for fact in facts:
    memory.remember(fact)

In [24]:
from pydantic import BaseModel
from typing import List
from operator import attrgetter
import builtins

class User(BaseModel):
    name: str
    score: int

users = [
    User(name="Alice", score=85),
    User(name="Bob", score=95),
    User(name="Charlie", score=70)
]

# 1. Get the full User object with the highest score
top_user = builtins.max(users, key=attrgetter('score'))
print(top_user.name)  # Output: Bob

# 2. Get just the highest score value
highest_score = builtins.max(u.score for u in users)
print(highest_score)  # Output: 95


Bob
95
